# 69. 交互气泡图

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 6 / 18 步：交互探索趋势、类别和变量关系**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 交互散点图（px.scatter）  →  **本章任务：** 交互气泡图  →  **下一步：** 交互面积图（px.area）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

分析业务数据时，我们常常要同时盯住好几个维度——销售额高不高、购买能力强不强，以及这类客户规模大不大。



## 本章目标

学完本章，你将能够：

- **理解**：理解「交互气泡图」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「交互气泡图」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「交互气泡图」并读出其中的结论。


## 69.1 适用场景

**背景引入**：分析业务数据时，我们常常要同时盯住好几个维度——销售额高不高、购买能力强不强，以及这类客户规模大不大。光靠一张表格很难把这些讲清楚，而交互式气泡图能用横轴、纵轴和气泡大小同时装下三路信息，鼠标悬停还能看到具体数值，让图表真正“会说话”，也让大家一眼看清谁是大客户、谁是潜力股。（好比地图上的城市圆点：位置是坐标，圆的大小是城市规模；但气泡要用“面积”代表数值，不是半径——若用半径，数值翻倍在视觉上会变成四倍，把差距夸大。）

同时比较X、Y和规模三个数值维度。


## 69.2 数据结构

两列位置变量、一列非负大小变量和可选分类字段。


## 69.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 size_max 从 70 改为 40 或 100，观察气泡最大直径对规模区分的影响
2. 移除 text 参数改用 hover_name，对比常显标签与悬浮标识的可读性
3. 修改 hovertemplate 自定义悬停信息格式，说明交互提示对规模精确读值的作用


## 69.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `orders.groupby()`、`px.scatter()`、`fig.update_layout()`、`fig.show()` | 同时比较X、Y和规模三个数值维度。 | 面积值跨度太大 |
| 进阶变体 | `orders.groupby()`、`px.scatter()`、`fig.update_layout()`、`fig.show()` | 在基础图表上增加分组、注释、布局或交互 | 最小气泡不可见 |
| 关键参数 | `size` | 气泡面积 | 面积值跨度太大 |
| 关键参数 | `size_max` | 最大直径 | 最小气泡不可见 |
| 关键参数 | `color` | 分类或连续值 | 用半径而非面积造成视觉误导 |
| 关键参数 | `hover_name` | 标识 | 面积值跨度太大 |


## 69.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-69 -->
### 数学推导｜气泡图应让面积而不是半径对应数值

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜圆的视觉面积由半径决定。** $A_i=\pi r_i^2$。

**第 2 步｜要求面积与数据成比例。** 设 $A_i=cv_i$，其中 $c$ 是统一缩放常数。

**第 3 步｜解出绘图半径。** 

$$
r_i=\sqrt{\frac{cv_i}{\pi}}
$$

如果错误地令 $r_i\propto v_i$，视觉面积会变成 $A_i\propto v_i^2$，大值会被夸张。

**把上面的关系收束为本章计算式：**

$$
A_i\propto v_i,\qquad r_i=\sqrt{\frac{A_i}{\pi}}\propto\sqrt{v_i}
$$

**符号解释：** $v_i$ 是编码数值，$A_i$ 是气泡面积，$r_i$ 是半径。

**代码对应：** 使用绘图库的 `size`/`sizeref` 机制，让视觉面积与数据量成比例。

**使用边界：** 气泡面积难以精确比较，应保留 Hover 数值并避免过大的尺寸跨度。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(f"Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行")


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 69.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
category_summary = orders.groupby("category", as_index=False).agg(
    order_value=("order_value", "mean"),
    items=("items", "mean"),
    sales=("sales", "sum"),
)
fig = px.scatter(
    category_summary,
    x="items",
    y="order_value",
    size="sales",
    color="category",
    text="category",
    size_max=70,
    title="品类规模气泡图",
)
fig.update_layout(
    xaxis_title="平均购买件数", yaxis_title="平均客单价（元）", showlegend=False
)
fig.show()


**练一练**：改一个参数，看气泡怎么变。

上面的 `px.scatter` 用 `size_max=70` 控制气泡的最大直径。请把它改成 `100`，重新运行并观察：气泡整体会明显变大，规模接近的品类更容易一眼区分，同时也能感受参数对图形信息密度的影响。


In [ ]:
# 请在下方填写代码：把 size_max 从 70 改成 100，重新生成气泡图。
# 提示：上方已算好的 category_summary 可以直接复用，只改 size_max 一个参数。

# 请把 None 替换为 px.scatter(...) 的返回值，并把 size_max 设为 100：
fig2 = None


In [ ]:
# 完整答案：把 size_max 改为 100，其余参数保持与基础图表一致
fig2 = px.scatter(
    category_summary,
    x="items",
    y="order_value",
    size="sales",
    color="category",
    text="category",
    size_max=100,  # 关键改动：最大气泡直径从 70 提高到 100
    title="品类规模气泡图（size_max=100）",
)
fig2.update_layout(
    xaxis_title="平均购买件数", yaxis_title="平均客单价（元）", showlegend=False
)
fig2.show()


## 69.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
region_category = orders.groupby(["region", "category"], as_index=False).agg(
    order_value=("order_value", "mean"),
    items=("items", "mean"),
    sales=("sales", "sum"),
)
fig = px.scatter(
    region_category,
    x="items",
    y="order_value",
    size="sales",
    color="region",
    hover_name="category",
    size_max=60,
    title="区域品类经营规模",
)
fig.update_layout(
    xaxis_title="平均购买件数", yaxis_title="平均客单价（元）", legend_title="区域"
)
fig.show()


## 69.8 参数说明

- size：气泡面积
- size_max：最大直径
- color：分类或连续值
- hover_name：标识


## 69.9 结果解读

位置优先于面积读取；面积只适合粗略比较规模。


## 69.10 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig = px.bar(report, x="region", y="sales", title="地区销售额")
fig.show()


### 69.10.1 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
fig.show()


### 69.10.2 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 69.11 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 69.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 69.12 易错点提醒

- 面积值跨度太大
- 最小气泡不可见
- 用半径而非面积造成视觉误导


## 69.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 69.14 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：换尺寸字段，用「订单量」替代「销售额」编码气泡大小
# 【目标】换一个 size 字段，看不同编码下气泡大小的含义变化。
import plotly.express as px

# 起点示例(已可运行)：size 换成 items(平均件数)，观察不同编码下的气泡。
category_summary = orders.groupby("category", as_index=False).agg(
    order_value=("order_value", "mean"), items=("items", "mean"), count=("sales", "count")
)
fig = px.scatter(
    category_summary,
    x="count",
    y="order_value",
    size="items",
    color="category",
    size_max=70,
    text="category",
    title="品类订单规模气泡图",
)
fig.update_layout(xaxis_title="订单量", yaxis_title="平均客单价（元）", showlegend=False)
fig.show()

# ---- 反思记录：换 size 编码后，气泡代表的含义有何变化 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
channel_summary = orders.groupby("channel", as_index=False).agg(
    order_value=("order_value", "mean"),
    items=("items", "mean"),
    sales=("sales", "sum"),
)
fig = px.scatter(
    channel_summary,
    x="order_value",
    y="items",
    size="sales",
    color="channel",
    text="channel",
    size_max=75,
    title="渠道价值与规模",
)
fig.update_layout(
    xaxis_title="平均客单价（元）", yaxis_title="平均购买件数", showlegend=False
)
fig.show()


## 69.15 小结

用气泡面积编码第三个数值变量，在二维位置上增加规模信息。


### 69.15.1 你已经掌握

- 判断交互气泡图的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 69.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `size` | 气泡面积 |
| `size_max` | 最大直径 |
| `color` | 分类或连续值 |
| `hover_name` | 标识 |


### 69.15.3 需要注意

- 面积值跨度太大
- 最小气泡不可见
- 用半径而非面积造成视觉误导


### 69.15.4 完成检查

- [ ] 能判断什么问题适合使用交互气泡图
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 69.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
